<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/3_0_BART_DRS_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook loads a pre-trained BART model, then attempts to recreate firms' Digital Readiness Scores (DRSs) via zero-shot classification using text about each firm.

This notebook runs best on a GPU runtime.

# Setup

In [ ]:
import os
import sys

The below settings appear to be necessary for successful downloading of the pre-trained models from HuggingFace.

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "15"

from huggingface_hub.utils import _runtime
_runtime._is_google_colab = False
HF_USE_TOKEN = False

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    repo_path = '/content/drive/MyDrive/digi-inno-road-prod'
    if os.path.isdir(repo_path):
      cwd = os.getcwd()
      os.chdir(repo_path)
      !git pull
      os.chdir(cwd)
    else:
      !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git /content/drive/MyDrive/digi-inno-road-prod
      print('Repository cloned into your Google Drive. It is strongly recommended that you copy the credentials.json, sheet.json, and token.json files into the secrets folder before proceeding.')
    sys.path.insert(0, repo_path)
    IN_COLAB = True
except ImportError:
    repo_path = os.path.abspath(os.path.join('../src'))
    IN_COLAB = False

if not repo_path in sys.path:
    sys.path.insert(0, repo_path)

In [ ]:
if IN_COLAB:
  output_path = os.path.join(repo_path, 'analysis', 'outputs')
else:
  output_path = os.path.join('.', 'outputs')
output_path = os.path.abspath(output_path)

# Import packages & data

Conda packages

In [ ]:
import math
import pandas
from transformers import pipeline

Project sourcecode packages

In [ ]:
from innoprod.digital_readiness_score import DRS_LEVELS
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps

Data

In [ ]:
data = get_sheet_dfs()

INCLUDE_NO_GRANTS_FIRMS = True
if INCLUDE_NO_GRANTS_FIRMS:
  roadmaps_df = pandas.concat([data['Roadmaps'], data['RoadmapsWithoutGrants']])
else:
  roadmaps_df = data['Roadmaps']
roadmaps_df = wrangle_roadmaps(roadmaps_df)

# Model parameters

In [ ]:
model_name = "facebook/bart-large-mnli"
hypothesis_template = ("This company's digital readiness level is best described as: {}")

In [ ]:
key_questions = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Details of any existing Digital Strategy',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis'
]

drs_col = 'Current Digital Readiness Score (refer to PAS:1040)'

# Transform data

In [ ]:
roadmaps_df = roadmaps_df[['Client ID', drs_col] + key_questions]

# Remove firms without a human-assigned DRS
roadmaps_df = roadmaps_df[roadmaps_df[drs_col].notna()]

# Clean the text responses to replace NaN and "nan" values with empty strings
roadmaps_df[key_questions] = roadmaps_df[key_questions].fillna('')
roadmaps_df[roadmaps_df[key_questions] == 'nan'] = ''

# Concatenate all the key questions into one, single string ("Context")
roadmaps_df['Context'] = roadmaps_df.apply(lambda row: ' '.join(filter(None, [row[c] for c in key_questions])), axis=1)
roadmaps_df.drop(key_questions, axis=1, inplace=True)

# Remove firms that have no (meaningful) text written about them
# roadmaps_df['Word Count'] = roadmaps_df.apply(lambda row: sum([len(str(row[q]).split()) for q in key_questions]), axis=1)
roadmaps_df['Word Count'] = roadmaps_df['Context'].str.split().str.len()
roadmaps_df = roadmaps_df[roadmaps_df['Word Count'] > 0].drop(columns=['Word Count'])

roadmaps_df = roadmaps_df.reset_index(drop=True)
# roadmaps_df

# Run pipeline

In [ ]:
classifier = pipeline("zero-shot-classification", model=model_name)

In [ ]:
context = qual_df['Context'].to_list()

results = classifier(
      context,
      candidate_labels=DRS_LEVELS,
      hypothesis_template=hypothesis_template
)

In [ ]:
roadmaps_df['Predicted DRS'] = [DRS_LEVELS.index(rd['labels'][0])+1 for rd in results]
roadmaps_df['Probability'] = [rd['scores'][0] for rd in results]
roadmaps_df['Confidence'] = [abs(math.log(rd['scores'][0]) - math.log(rd['scores'][1])) for rd in results]
roadmaps_df = roadmaps_df.drop(columns='Context')
#roadmaps_df

# Write results

In [ ]:
roadmaps_df.to_csv(os.path.join(output_path, 'BART_DRS_results.csv'), index=False)